# Superstore SQL Analysis — Notebook
**Dataset:** Sample Superstore (9,994 rows)  
**Dialect:** SQLite via `sqlite3` + `pandas`  
**Techniques:** Subqueries · CTEs · Window Functions · JOINs  
**Sections:**
1. Setup & Load
2. Data Cleaning
3. Schema Creation (customers / products / orders)
4. Subqueries
5. CTEs
6. Window Functions
7. Combined: JOIN + CTE + Window Functions
8. Business Queries
9. Brief Insights


## 1. Setup & Load

In [ ]:
import sqlite3
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

DB_PATH = "superstore_analysis.db"

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
print("Connected to:", DB_PATH)

df_raw = pd.read_sql("SELECT * FROM superstore_raw", conn)
print(f"Loaded superstore_raw: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)


## 2. Data Cleaning
Steps applied:
- Parse `order_date` and `ship_date` to `datetime`
- Derive `days_to_ship` (integer)
- Normalise `postal_code` to zero-padded 5-char string
- Add `profit_margin_pct` column
- Flag `high_discount` rows (discount > 0.5)
- Flag `loss_order` rows (profit < 0)
- Export cleaned CSV


In [ ]:
df = df_raw.copy()

# 1. Parse dates
df["order_date"] = pd.to_datetime(df["order_date"], format="%m/%d/%Y")
df["ship_date"]  = pd.to_datetime(df["ship_date"],  format="%m/%d/%Y")

# 2. Derived: days to ship
df["days_to_ship"] = (df["ship_date"] - df["order_date"]).dt.days

# 3. Normalise postal code to 5-char zero-padded string
df["postal_code"] = df["postal_code"].astype(str).str.zfill(5)

# 4. Profit margin %
df["profit_margin_pct"] = (df["profit"] / df["sales"] * 100).round(2)

# 5. Flag columns
df["high_discount"] = df["discount"] > 0.5
df["loss_order"]    = df["profit"] < 0

print("Cleaning summary")
print("-" * 40)
print(f"  Rows              : {len(df):,}")
print(f"  Nulls (any col)   : {df.isnull().sum().sum()}")
print(f"  Duplicate rows    : {df.duplicated().sum()}")
print(f"  Loss orders       : {df['loss_order'].sum():,}  ({df['loss_order'].mean()*100:.1f}%)")
print(f"  High-discount rows: {df['high_discount'].sum():,}  ({df['high_discount'].mean()*100:.1f}%)")
print(f"  Avg days to ship  : {df['days_to_ship'].mean():.1f}")
print(f"  Date range        : {df['order_date'].min().date()} → {df['order_date'].max().date()}")
df.head(3)


In [ ]:
# Export cleaned CSV
CLEAN_CSV = "superstore_cleaned.csv"
df.to_csv(CLEAN_CSV, index=False)
print(f"Saved cleaned CSV → {CLEAN_CSV}  ({len(df):,} rows)")


## 3. Schema Creation

In [ ]:
# Verify the three derived tables already in the DB
for tbl in ("customers", "products", "orders"):
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {tbl}", conn).iloc[0, 0]
    print(f"  {tbl:<12} {n:>5,} rows")


In [ ]:
# Preview each table
for tbl in ("customers", "products", "orders"):
    print(f"\n── {tbl} ──")
    display(pd.read_sql(f"SELECT * FROM {tbl} LIMIT 3", conn))


## 4. Subqueries
### 4a. Order lines above average sales

In [ ]:
SQL_ABOVE_AVG = """
SELECT
    o.order_id,
    c.customer_name,
    p.product_name,
    ROUND(o.sales, 2) AS sales
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
JOIN products  p ON p.product_id  = o.product_id
WHERE o.sales > (SELECT AVG(sales) FROM orders)
ORDER BY o.sales DESC
LIMIT 10
"""
df_above = pd.read_sql(SQL_ABOVE_AVG, conn)
print(f"Top 10 of {pd.read_sql('SELECT COUNT(*) n FROM orders WHERE sales>(SELECT AVG(sales) FROM orders)',conn).iloc[0,0]:,} above-avg lines")
df_above


### 4b. Highest single order line per customer

In [ ]:
SQL_MAX_PER_CUST = """
SELECT
    c.customer_name,
    o.order_id,
    p.product_name,
    ROUND(o.sales, 2) AS sales
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
JOIN products  p ON p.product_id  = o.product_id
WHERE o.sales = (
    SELECT MAX(o2.sales)
    FROM orders o2
    WHERE o2.customer_id = o.customer_id
)
ORDER BY o.sales DESC
LIMIT 10
"""
pd.read_sql(SQL_MAX_PER_CUST, conn)


## 5. CTEs (Common Table Expressions)
### 5a. Top 10 customers by total sales

In [ ]:
SQL_TOP_CUST = """
WITH customer_sales AS (
    SELECT
        customer_id,
        ROUND(SUM(sales), 2)        AS total_sales,
        COUNT(DISTINCT order_id)    AS order_count
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name,
    cs.total_sales,
    cs.order_count
FROM customer_sales cs
JOIN customers c ON c.customer_id = cs.customer_id
ORDER BY cs.total_sales DESC
LIMIT 10
"""
pd.read_sql(SQL_TOP_CUST, conn)


### 5b. Bottom 10 customers by total sales

In [ ]:
SQL_LOW_CUST = SQL_TOP_CUST.replace("DESC", "ASC")
pd.read_sql(SQL_LOW_CUST, conn)


### 5c. Single-order customers

In [ ]:
SQL_SINGLE = """
WITH order_counts AS (
    SELECT
        customer_id,
        COUNT(DISTINCT order_id)    AS order_count,
        ROUND(SUM(sales), 2)        AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name,
    oc.order_count,
    oc.total_sales
FROM order_counts oc
JOIN customers c ON c.customer_id = oc.customer_id
WHERE oc.order_count = 1
ORDER BY oc.total_sales DESC
"""
df_single = pd.read_sql(SQL_SINGLE, conn)
print(f"{len(df_single)} single-order customers")
df_single


## 6. Window Functions
### 6a. Product sales rank within each category (RANK + ROW_NUMBER)

In [ ]:
SQL_PROD_RANK = """
SELECT
    p.category,
    p.product_name,
    ROUND(SUM(o.sales), 2) AS product_sales,
    RANK() OVER (
        PARTITION BY p.category
        ORDER BY SUM(o.sales) DESC
    ) AS category_sales_rank,
    ROW_NUMBER() OVER (
        PARTITION BY p.category
        ORDER BY SUM(o.sales) DESC, p.product_id
    ) AS category_row_num
FROM orders o
JOIN products p ON p.product_id = o.product_id
GROUP BY p.category, p.product_id, p.product_name
ORDER BY p.category, category_sales_rank
LIMIT 15
"""
pd.read_sql(SQL_PROD_RANK, conn)


## 7. Combined — JOIN + CTE + Window Functions

In [ ]:
SQL_COMBINED = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
),
ranked_customers AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.segment,
        ROUND(cs.total_sales, 2)                                        AS total_sales,
        RANK()       OVER (ORDER BY cs.total_sales DESC)                AS sales_rank,
        ROW_NUMBER() OVER (ORDER BY cs.total_sales DESC, c.customer_id) AS row_num
    FROM customer_sales cs
    JOIN customers c ON c.customer_id = cs.customer_id
)
SELECT
    customer_name,
    segment,
    total_sales,
    sales_rank,
    row_num
FROM ranked_customers
ORDER BY sales_rank
LIMIT 10
"""
pd.read_sql(SQL_COMBINED, conn)


## 8. Business Queries on Cleaned Data
Using the enriched `df` (with `days_to_ship`, `profit_margin_pct`, etc.)

### 8a. Average days to ship by ship mode

In [ ]:
df.groupby("ship_mode")["days_to_ship"].agg(["mean","min","max"]).round(2).sort_values("mean")


### 8b. Profit margin by category

In [ ]:
df.groupby("category").agg(
    total_sales=("sales","sum"),
    total_profit=("profit","sum"),
    avg_margin_pct=("profit_margin_pct","mean")
).round(2).sort_values("total_profit", ascending=False)


### 8c. High-discount impact on profit

In [ ]:
df.groupby("high_discount").agg(
    rows=("profit","count"),
    avg_profit=("profit","mean"),
    avg_discount=("discount","mean"),
    pct_loss=("loss_order","mean")
).round(3).rename(index={True:"High discount (>0.5)", False:"Normal discount"})


### 8d. Annual sales trend

In [ ]:
df["year"] = df["order_date"].dt.year
df.groupby("year").agg(
    total_sales=("sales","sum"),
    total_profit=("profit","sum"),
    orders=("order_id","nunique")
).round(2)


## 9. Brief Insights


In [ ]:
avg_sales   = df["sales"].mean()
total_sales = df["sales"].sum()
total_profit= df["profit"].sum()
above_avg   = (df["sales"] > avg_sales).sum()
single_ord  = df.groupby("customer_id")["order_id"].nunique()
single_cnt  = (single_ord == 1).sum()

print("=" * 50)
print("  SUPERSTORE KEY METRICS")
print("=" * 50)
print(f"  Total sales          : ${total_sales:>12,.2f}")
print(f"  Total profit         : ${total_profit:>12,.2f}")
print(f"  Overall margin       :  {total_profit/total_sales*100:>10.1f}%")
print(f"  Avg order-line sales : ${avg_sales:>12,.2f}")
print(f"  Above-avg lines      :  {above_avg:>10,}  of 9,994")
print(f"  Single-order cust.   :  {single_cnt:>10,}")
print(f"  Loss-making lines    :  {df['loss_order'].sum():>10,}  ({df['loss_order'].mean()*100:.1f}%)")
print(f"  High-discount lines  :  {df['high_discount'].sum():>10,}  ({df['high_discount'].mean()*100:.1f}%)")
print(f"  Avg days to ship     :  {df['days_to_ship'].mean():>10.1f}")
print("=" * 50)
print()
print("Key findings:")
print("  • Sean Miller tops sales at $25,043 across 5 orders.")
print("  • Canon imageCLASS 2200 Copier dominates top single-order lines.")
print("  • 18.7% of lines are loss-making; 8.6% carry >50% discount.")
print("  • High-discount lines have significantly lower avg profit.")
print("  • 12 customers placed exactly one order — a retention opportunity.")


In [ ]:
conn.close()
print('Connection closed.')